In [1]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


C:\Users\Paul Dean\PyCharmMiscProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hidden_set = pd.read_csv("hidden_test_with_labels.csv")
model = AutoModelForSequenceClassification.from_pretrained("model_checkpoint/")
tokenizer = AutoTokenizer.from_pretrained("model_checkpoint/")

inputs = tokenizer(hidden_set['text'].tolist(),padding="max_length", max_length=512, truncation=True, return_tensors="pt")



model.eval()
predictions = []
batch_size = 16
with torch.no_grad():
    for i in range(0, len(hidden_set), batch_size):
        batch = {k: v[i:i+batch_size] for k, v in inputs.items()}
        outputs = model(**batch)
        predictions.extend(outputs.logits.argmax(-1).tolist())

print(classification_report(hidden_set['label'], predictions, target_names=["negative", "positive"]))
print(confusion_matrix(hidden_set['label'], predictions))
predictions_df = pd.DataFrame({
    "id": hidden_set["id"],
    "predicted_label": predictions
})
predictions_df.to_csv("hidden_test_predictions.csv", index=False)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4613.92it/s]


              precision    recall  f1-score   support

    negative       0.94      0.55      0.69       300
    positive       0.68      0.97      0.80       300

    accuracy                           0.76       600
   macro avg       0.81      0.76      0.75       600
weighted avg       0.81      0.76      0.75       600

[[164 136]
 [ 10 290]]


# Reflection

Hidden test accuracy:

                precision    recall  f1-score   support

    negative       0.94      0.55      0.69       300
    positive       0.68      0.97      0.80       300

    accuracy                           0.76       600
    macro avg      0.81      0.76      0.75       600
    weighted avg   0.81      0.76      0.75       600

Hidden confusion Matrix:
$\begin{bmatrix}164 & 136\\10 & 290\end{bmatrix}$

### Comparison:
Both tests on public and hidden are similar in accuracy with the public being a 75% accuracy and the hidden being a 76% accuracy showing that the model is consistent across different test sets.The model increased its overcorrection by making more negatives positive however when the model does predict that it is positive then it is highly precise.

### Time and Computing
First of all if I had more time I would have tweaked the hyperparameters further such as the learning weight as well as testing different models which would also benefit from more computing power by taking a stronger pretrained model and forming it to the training set. More time would probably have resulted in a slightly higher accuracy however the limited training set makes massive improvements difficult.